In [1]:
import numpy as np
import pandas as pd
import transformers
import torch

from tqdm import tqdm, trange
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, Add
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report


In [5]:
df = pd.read_csv('/content/drive/MyDrive/CAN_Research/CAN/attack-free-1.csv', delimiter=',')
df['datetime'] = df['timestamp']
df['arbitration_id'] = df['arbitration_id'].astype(str).dropna().apply(lambda x: float.fromhex(x))
df['data_field'] = df['data_field'].astype(str).dropna().apply(lambda x: float.fromhex(x))
df['attack'] = 0
df = df.loc[:50000, ['datetime','arbitration_id', 'data_field', 'attack']]


df.head(5)

,datetime,arbitration_id,data_field,attack
0,1.672531e+09,193.0,3.458765e+18,0
1,1.672531e+09,197.0,3.458765e+18,0
2,1.672531e+09,388.0,8.589935e+09,0
3,1.672531e+09,455.0,1.790802e+15,0
4,1.672531e+09,461.0,0.000000e+00,0


In [7]:
dg = pd.read_csv('/content/drive/MyDrive/CAN_Research/CAN/DoS-1.csv', delimiter=',')
dg['datetime'] = dg['timestamp']
dg['arbitration_id'] = dg['arbitration_id'].astype(str).dropna().apply(lambda x: float.fromhex(x))
dg['data_field'] = dg['data_field'].astype(str).dropna().apply(lambda x: float.fromhex(x))
dg['attack'] = 1
dg = dg.loc[:50000, ['datetime','arbitration_id', 'data_field', 'attack']]

dg.head(5)

,datetime,arbitration_id,data_field,attack
0,1.672531e+09,485.0,5.044479e+18,1
1,1.672531e+09,489.0,4.503651e+15,1
2,1.672531e+09,249.0,1.225120e+17,1
3,1.672531e+09,761.0,6.313770e+11,1
4,1.672531e+09,409.0,1.498772e+19,1


In [8]:
# Feature and label extraction
from sklearn.utils import shuffle
data = pd.concat([df, dg])
data = shuffle(data)

X = data[['arbitration_id', 'data_field']]
y = data['attack']

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

input_layer = Input(shape=(X_train.shape[1],))
positional_encoding = Dense(32, activation='relu')(input_layer)
attention_output = MultiHeadAttention(num_heads=4, key_dim=32)(positional_encoding, positional_encoding)
attention_output = Add()([positional_encoding, attention_output])
attention_output = LayerNormalization()(attention_output)
attention_output = Dropout(0.15)(attention_output)
feed_forward = Dense(64, activation='relu')(attention_output)
feed_forward = Dropout(0.15)(feed_forward)
output_layer = Dense(1, activation='sigmoid')(feed_forward)

model = Model(inputs=input_layer, outputs=output_layer)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])





IndexError: Exception encountered when calling layer 'softmax' (type Softmax).

tuple index out of range

Call arguments received by layer 'softmax' (type Softmax):
  • inputs=tf.Tensor(shape=(None, 4), dtype=float32)
  • mask=None